# 02 - Agent loop with OpenAI-compatible tool calling

The agents in this project are language models that answer questions about Hetionet by writing Cypher. This notebook shows the tool-calling loop from `src/agent.py` (built on the `openai` SDK): what the model is given, and one logged run replayed step by step.

The replay needs no API key. The last cell calls a model only if an endpoint is configured.

In [1]:
import os
import sys
from pathlib import Path

# Find the repository root (the folder with pyproject.toml), wherever the notebook is started from.
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").exists())
os.chdir(ROOT)                              # project code uses paths relative to the repo root
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))   # modules in src/ import each other flat: `from db import run_cypher`
print("repo root:", ROOT.name)

repo root: reproducible-agents-biomedical-kg


In [2]:
import json

import agent                       # src/agent.py
from prompt import SYSTEM_PROMPT   # src/prompt.py

print("Tool definition sent with every request (agent.TOOLS):")
print(json.dumps(agent.TOOLS, indent=2))

prompt_lines = SYSTEM_PROMPT.splitlines()
print(f"\nSYSTEM_PROMPT, first 15 of {len(prompt_lines)} lines:")
print("\n".join(prompt_lines[:15]))

Tool definition sent with every request (agent.TOOLS):
[
  {
    "type": "function",
    "function": {
      "name": "run_cypher",
      "description": "Run a read-only Cypher query (Neo4j 3.5) on the Hetionet graph. Returns columns, rows, total_rows, truncated, or an error message.",
      "parameters": {
        "type": "object",
        "properties": {
          "query": {
            "type": "string",
            "description": "The Cypher query"
          }
        },
        "required": [
          "query"
        ]
      }
    }
  }
]

SYSTEM_PROMPT, first 15 of 22 lines:
You answer questions about Hetionet, a biomedical knowledge graph stored in Neo4j 3.5.
Use the run_cypher tool to query it. Base every answer on query results, never on memory.

Cypher dialect: Neo4j 3.5. Do not use newer syntax such as EXISTS { ... } subqueries.
Queries are read-only. Results are capped at 50 rows; check the 'truncated' flag.

Node labels and key property (all have name and identifier):
Compou

## The loop

1. The model receives the system prompt, the question and the definition of one tool, `run_cypher`.
2. Instead of text it can reply with a tool call: the tool name and JSON arguments holding a Cypher query.
3. Our code runs the query with `db.run_cypher` and appends the result (rows or an error message) as a `tool` message.
4. This repeats until the model answers in plain text, or after 6 model calls (`max_steps=6`).

In the SDK this is `client.chat.completions.create(model=..., messages=..., tools=TOOLS, temperature=0.0, seed=0)`. The reply has either `message.tool_calls` (steps 2 and 3) or `message.content` (the final answer).

## Replay mode: a logged run

`results/pilot_20261025_131041.jsonl` holds the pilot runs of 5 models on 5 questions. Each record stores every tool call (`steps`), the final answer, token counts and latency, so a run can be inspected without calling a model.

In [3]:
import ast

LOG = Path("results/pilot_20261025_131041.jsonl")
records = [json.loads(line) for line in LOG.read_text(encoding="utf-8-sig").splitlines() if line.strip()]


def parsed(value):
    """Nested fields are JSON in this log; a Python-literal string is accepted too."""
    return ast.literal_eval(value) if isinstance(value, str) else value


def failed(step):
    return "error" in step["result"]


runs = [r for r in records if "steps" in r]   # runs that stopped on an infrastructure error may have no steps
for r in runs:
    r["steps"] = parsed(r["steps"])
print(f"{len(records)} records, {len(runs)} with logged steps, "
      f"{sum('infra_error' in r for r in records)} marked as infrastructure errors")

# Runs whose first query was rejected by the server and whose last query succeeded.
recovered = [r for r in runs if len(r["steps"]) >= 2 and failed(r["steps"][0]) and not failed(r["steps"][-1])]
print("recovered after a failed first query:", [(r["model"], r["qid"]) for r in recovered])
rec = recovered[0]

25 records, 23 with logged steps, 3 marked as infrastructure errors
recovered after a failed first query: [('mistral-small-24b', 'q04')]


In [4]:
import textwrap


def show_run(run, max_rows=3):
    """Print an agent run: each Cypher query sent, what the tool returned, then the final answer."""
    for i, step in enumerate(run["steps"], 1):
        print(f"--- step {i}: the model calls run_cypher with")
        print(textwrap.fill(step["cypher"], 100, initial_indent="    ", subsequent_indent="    "))
        result = step["result"]
        if "error" in result:
            print("    tool result: ERROR", result["error"].strip().split("\n")[0])
        else:
            print(f"    tool result: {result['total_rows']} rows, columns {result['columns']}, "
                  f"truncated={result['truncated']}")
            for row in result["rows"][:max_rows]:
                print("       ", row)
            if result["total_rows"] > max_rows:
                print(f"        ... {result['total_rows'] - max_rows} more")
    print("--- final answer")
    print(run["final_answer"])
    print(f"--- {run['n_steps']} tool calls | {run['tokens']} tokens (prompt {run.get('prompt_tokens')}, "
          f"completion {run.get('completion_tokens')}) | {run['latency_s']} s")


lines = Path("data/questions.jsonl").read_text(encoding="utf-8-sig").splitlines()
questions = [json.loads(line) for line in lines if line.strip()]
question = next(q for q in questions if q["id"] == rec["qid"])

print("model   :", rec["model"], f"(model id {rec['model_id']})")
print("question:", textwrap.fill(f"{rec['qid']} - {question['question']}", 100, subsequent_indent=" " * 10))
show_run(rec)
print("scored against the gold answer: exact =", rec["exact"])

model   : mistral-small-24b (model id Mistral-Small-3.2:24b)
question: q04 - Which 5 compounds bind the most genes associated with osteoporosis but are NOT known to treat
          it? Return name and shared gene count; break ties alphabetically by name.
--- step 1: the model calls run_cypher with
    MATCH (d:Disease {name: 'osteoporosis'})-[:ASSOCIATES_DaG]->(g:Gene)<-[:BINDS_CbG]-(c:Compound)
    WHERE NOT (c)-[:TREATS_CtD]->(d) WITH c, COUNT(g) AS shared_genes COUNT DISTINCT g ORDER BY
    shared_genes DESC, c.name ASC LIMIT 5 RETURN c.name AS compound, shared_genes
    tool result: ERROR Invalid input 'U': expected 'n/N' (line 1, column 165 (offset: 164))
--- step 2: the model calls run_cypher with
    MATCH (d:Disease {name: 'osteoporosis'})-[:ASSOCIATES_DaG]->(g:Gene)<-[:BINDS_CbG]-(c:Compound)
    WHERE NOT (c)-[:TREATS_CtD]->(d) WITH c, COUNT(g) AS shared_genes ORDER BY shared_genes DESC,
    c.name ASC LIMIT 5 RETURN c.name AS compound, shared_genes
    tool result: 5 rows, c

## Live mode

Runs the same loop against a real model. It needs `LLM_BASE_URL` and `LLM_API_KEY` (see `.env.example`) and is skipped without them.

In [5]:
import os

from models import MODELS   # src/models.py: model ids; endpoint and key come from the environment

model_name, cfg = next(iter(MODELS.items()))
question = questions[0]["question"]

if os.getenv("LLM_BASE_URL") and os.getenv("LLM_API_KEY"):
    print("model:", model_name, "| question:", question)
    live = agent.run_agent(cfg, question)   # defaults: max_steps=6, temperature=0.0, seed=0
    show_run(live)
else:
    print("skipped: no API key configured")

skipped: no API key configured


## Takeaways

- The model never touches the database. It only emits a `run_cypher` call with a query; our code runs it and returns the result as a `tool` message.
- Errors go back to the model as data. In the replayed run the first query had a stray `COUNT DISTINCT g`, the server answered with a syntax error, and the second query was correct.
- Every model gets the same prompt, tool, temperature 0 and seed; the log field `seed_used` records whether the provider accepted the seed. Each step, token count and latency is logged, so runs can be compared and replayed without calling the API again.